# Fine-Tuning Llama-3.1-8B con Weighted Loss (QLoRA)

**Estrategia:** Dataset completo + Weighted Cross-Entropy Loss para manejar desbalanceo

**CONFIGURACIÓN ÓPTIMA - BALANCE VELOCIDAD/CALIDAD:**
- Batch size = 16 con gradient accumulation = 2 (batch efectivo = 32)
- QLoRA (4-bit quantization NF4)
- BF16 mixed precision FULL (train + eval)
- LoRA rank reducido (r=16)
- Max length = 256 tokens
- Evaluaciones cada 1000 steps
- Gradient checkpointing OFF
- 4 workers + prefetch=2
- Optimizer adamw_fused
- Warmup 5%

**Tiempo estimado: 6-8 horas**

**VRAM: 12-13GB (uso eficiente del hardware)**

**CALIDAD GARANTIZADA:** Mismo batch efectivo (32) que notebook 12 original

**Diferencias vs Notebook 12:**
- Mismo modelo (Llama-3.1-8B-Instruct)
- Mismo prompt de instrucción
- Mismo batch efectivo = 32 (calidad idéntica)
- SIN undersampling (dataset completo: 156,695 muestras)
- CON weighted loss (pos_weight aprox. 4.17x)
- Checkpoints en directorio separado (llama-3.1-8b-weighted)
- Entrenamiento 3-4x más rápido que config original

## 1. Imports y Configuración

In [ ]:
# Imports necesarios para fine-tuning con QLoRA y weighted loss
# - transformers: modelo base Llama-3.1-8B + Trainer API
# - peft: LoRA adapters para entrenar solo ~0.17% de parámetros
# - bitsandbytes: quantización 4-bit para reducir VRAM
import torch
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType, PeftModel
from datasets import Dataset
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings('ignore')

# Verificar disponibilidad de GPU (RTX 5070 Ti con 16GB VRAM)
print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 2. Definir Paths y Configuración

In [ ]:
# Configuración optimizada para RTX 5070 Ti (16GB VRAM)
# Objetivo: Balance entre velocidad de entrenamiento y calidad de resultados

# Directorios de datos y checkpoints
BASE_DIR = Path("/home/eeguskiza/DEUSTO/multi-author-analysis")
DATA_RAW = BASE_DIR / "data" / "raw"  # Textos originales por oración
BOUNDARIES_DIR = BASE_DIR / "data" / "processed" / "boundaries"  # CSVs con pares y labels
CHECKPOINT_DIR = BASE_DIR / "checkpoints" / "finetuning" / "llama-3.1-8b-weighted"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Modelo base: Llama-3.1-8B-Instruct (instrucción preentrenada)
MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# CONFIGURACIÓN OPTIMIZADA
# - batch_size=16: máximo sin OOM en RTX 5070 Ti
# - gradient_accumulation=2: batch efectivo=32 (igual calidad que notebook 12)
# - lora_r=16: rank bajo para reducir parámetros entrenables
# - max_length=256: suficiente para 2 oraciones concatenadas
CONFIG = {
    "model_name": MODEL_NAME,
    "batch_size": 16,  # Batch físico por GPU
    "gradient_accumulation_steps": 2,  # Batch efectivo = 32
    "learning_rate": 2e-4,  # Learning rate estándar para LoRA
    "epochs": 1,  # 1 época con dataset completo (156k muestras)
    "lora_r": 16,  # LoRA rank (dimensión de adaptadores)
    "lora_alpha": 32,  # LoRA alpha (escala de adaptadores)
    "lora_dropout": 0.1,  # Dropout para regularización
    "target_modules": ["q_proj", "v_proj"],  # Módulos de atención a adaptar
    "max_length": 256,  # Longitud máxima en tokens
}

print(f"Configuración cargada:")
print(f"  - Batch size efectivo: {CONFIG['batch_size'] * CONFIG['gradient_accumulation_steps']}")
print(f"  - Learning rate: {CONFIG['learning_rate']}")
print(f"  - Max length: {CONFIG['max_length']} tokens")
print(f"  - Checkpoints: {CHECKPOINT_DIR}")
print(f"\nCONFIGURACIÓN ÓPTIMA PARA TU HARDWARE")
print(f"  - Batch size: {CONFIG['batch_size']} (óptimo rendimiento)")
print(f"  - Gradient accumulation: {CONFIG['gradient_accumulation_steps']}")
print(f"  - Batch efectivo: 32 (MISMA CALIDAD que notebook 12)")
print(f"  - Max length: {CONFIG['max_length']}")
print(f"  - Evaluaciones cada 1000 steps")
print(f"  - Gradient checkpointing: OFF")
print(f"  - Workers: 4 threads paralelos")
print(f"  - Optimizer: adamw_fused")
print(f"\nVRAM esperada: 12-13GB (uso eficiente)")
print(f"Total steps: ~{156695 // 32:,}")
print(f"\nCALIDAD: GARANTIZADA (batch efectivo = 32, igual que notebook 12)")
print(f"TIEMPO ESTIMADO: 6-8 horas")

## 3. Cargar Dataset Completo (Sin Undersampling)

In [ ]:
# Funciones para cargar dataset completo desde archivos raw
# Estrategia: Sin undersampling, se usa weighted loss para manejar desbalanceo

def load_sentences(level: str, split: str, doc_id: str) -> list:
    """
    Carga las oraciones de un documento desde el archivo raw.
    Los archivos están en formato texto plano, una oración por línea.
    
    Args:
        level: 'easy', 'medium', o 'hard' (dificultad del problema)
        split: 'train' o 'validation'
        doc_id: ID del documento (ej: 'problem-1')
        
    Returns:
        list[str]: Lista de oraciones del documento
    """
    path = DATA_RAW / level / split / f"{doc_id}.txt"
    
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")
    
    # Leer y dividir por líneas (cada línea es una oración)
    text = path.read_text(encoding='utf-8').strip()
    sentences = text.split('\n')
    
    # Filtrar líneas vacías
    sentences = [s.strip() for s in sentences if s.strip()]
    
    return sentences


def create_text_pairs(df: pd.DataFrame, desc: str = "Creando pares") -> list:
    """
    Crea lista de pares de oraciones con labels desde el CSV de boundaries.
    Lee los textos originales usando los índices del CSV.
    
    El CSV contiene:
    - level, split, doc_id: identificación del documento
    - sent_left_id, sent_right_id: índices de las oraciones consecutivas
    - y: label (0=mismo autor, 1=cambio de autor)
    
    Args:
        df: DataFrame con boundaries (level, split, doc_id, sent_left_id, sent_right_id, y)
        desc: Descripción para tqdm
        
    Returns:
        list[dict]: Lista de diccionarios con pares de oraciones y labels
    """
    pairs = []
    skipped = 0
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        try:
            # Cargar oraciones del documento
            sentences = load_sentences(row['level'], row['split'], row['doc_id'])
            
            # Validar que los índices existan
            if row['sent_left_id'] >= len(sentences) or row['sent_right_id'] >= len(sentences):
                skipped += 1
                continue
            
            # Obtener las dos oraciones consecutivas
            text_a = sentences[row['sent_left_id']]
            text_b = sentences[row['sent_right_id']]
            
            # Validar que no estén vacías
            if not text_a or not text_b:
                skipped += 1
                continue
            
            # Añadir el par al dataset
            pairs.append({
                'text_a': text_a,
                'text_b': text_b,
                'label': int(row['y']),  # 0=mismo autor, 1=cambio autor
                'level': row['level'],
                'doc_id': row['doc_id']
            })
            
        except Exception as e:
            skipped += 1
            if skipped <= 5:  # Mostrar solo los primeros errores
                print(f"\nError en {row['doc_id']}: {e}")
    
    if skipped > 0:
        print(f"\nSe saltaron {skipped} ejemplos por errores o índices inválidos")
    
    return pairs


# Cargar CSVs de boundaries (contienen índices de pares y labels)
print("Cargando datasets...")
boundaries_train = pd.read_csv(BOUNDARIES_DIR / "boundaries_train.csv")
boundaries_val = pd.read_csv(BOUNDARIES_DIR / "boundaries_validation.csv")

print(f"\nDataset COMPLETO (sin undersampling):")
print(f"  - Train: {len(boundaries_train)} muestras")
print(f"  - Val: {len(boundaries_val)} muestras")

# Crear pares de textos con labels
train_pairs = create_text_pairs(boundaries_train, desc="Creando pares de entrenamiento")
val_pairs = create_text_pairs(boundaries_val, desc="Creando pares de validación")

# Convertir a HuggingFace Dataset
train_dataset = Dataset.from_list(train_pairs)
val_dataset = Dataset.from_list(val_pairs)

print(f"\nDatasets creados:")
print(f"  - Train: {len(train_dataset)} pares")
print(f"  - Val: {len(val_dataset)} pares")

## 4. Calcular Ratio de Desbalanceo (pos_weight)

In [ ]:
# Calcular pos_weight para manejar desbalanceo de clases con weighted loss
# El dataset tiene ~80% clase 0 (mismo autor) y ~20% clase 1 (cambio autor)
# pos_weight = num_negativos / num_positivos (penaliza más errores en clase minoritaria)

# Contar ejemplos por clase
labels_train = np.array([sample['label'] for sample in train_dataset])
num_negativos = np.sum(labels_train == 0)  # Clase 0: mismo autor
num_positivos = np.sum(labels_train == 1)  # Clase 1: cambio de autor

# Calcular pos_weight para weighted cross-entropy
# pos_weight se aplica a la clase 1 (positiva/minoritaria)
# Un pos_weight=4.17 significa que errores en clase 1 pesan 4.17x más en la loss
pos_weight = num_negativos / num_positivos

print("="*60)
print("ANÁLISIS DE DESBALANCEO")
print("="*60)
print(f"Clase 0 (mismo autor):  {num_negativos:,} muestras ({num_negativos/len(labels_train)*100:.2f}%)")
print(f"Clase 1 (cambio autor): {num_positivos:,} muestras ({num_positivos/len(labels_train)*100:.2f}%)")
print(f"\nRATIO DE DESBALANCEO (pos_weight): {pos_weight:.4f}")
print(f"  → La clase 1 será penalizada {pos_weight:.2f}x más en la loss function")
print("="*60)

## 5. Cargar Tokenizer

In [ ]:
# Cargar tokenizer de Llama-3.1-8B
# Llama no tiene pad_token por defecto, se usa eos_token como pad_token
print("Cargando tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Configurar pad token (necesario para batching)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Tokenizer cargado. Vocab size: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")

## 6. Tokenizar Datasets

In [ ]:
# Tokenizar datasets con prompt de instrucción
# Formato: Instrucción + Text A + Text B + Pregunta
# Mismo prompt que notebook 12 original para comparabilidad

def tokenize_function(examples):
    """
    Tokeniza pares de textos con formato de instrucción para Llama.
    USA EL MISMO PROMPT QUE EL NOTEBOOK 12 ORIGINAL.
    
    El prompt explica la tarea y presenta las dos oraciones consecutivas.
    El modelo debe aprender a clasificar si son del mismo autor (0) o no (1).
    """
    texts = []
    for text_a, text_b in zip(examples['text_a'], examples['text_b']):
        # PROMPT EXACTO DEL NOTEBOOK 12 (para comparabilidad)
        prompt = (
            f"Analyze the writing style of these two consecutive sentences and determine if they were written by the same author.\n\n"
            f"Text A: {text_a}\n"
            f"Text B: {text_b}\n\n"
            f"Are these sentences from the same author?"
        )
        texts.append(prompt)
    
    # Tokenizar con padding a longitud fija y truncamiento
    return tokenizer(
        texts,
        padding='max_length',  # Padding a max_length para batching eficiente
        truncation=True,  # Truncar si supera max_length
        max_length=CONFIG['max_length'],  # 256 tokens
        return_tensors=None
    )

print("Tokenizando datasets...")
train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True,
    desc="Tokenizando train",
    remove_columns=['text_a', 'text_b', 'level', 'doc_id']  # Solo mantener input_ids, attention_mask, label
)

val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True,
    desc="Tokenizando val",
    remove_columns=['text_a', 'text_b', 'level', 'doc_id']
)

print(f"\nTokenización completada:")
print(f"  - Train: {len(train_tokenized)} muestras")
print(f"  - Val: {len(val_tokenized)} muestras")
print(f"  - Max length: {CONFIG['max_length']} tokens")

## 7. Configurar Modelo con QLoRA (4-bit)

In [ ]:
# Configurar modelo con quantización 4-bit (QLoRA)
# QLoRA = Quantized Low-Rank Adaptation
# - Quantización 4-bit: reduce VRAM de ~32GB a ~12GB
# - NF4: Normal Float 4-bit (distribución gaussiana optimizada para pesos)
# - BF16 compute: usa bfloat16 para cálculos (mejor precision que FP16 en Ada Lovelace)
# - Double quantization: quantiza también los parámetros de quantización

print("Configurando BitsAndBytes (4-bit quantization)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Cargar modelo en 4-bit
    bnb_4bit_quant_type="nf4",  # NF4 (Normal Float 4)
    bnb_4bit_compute_dtype=torch.bfloat16,  # BF16 para cálculos (óptimo en RTX 5070 Ti)
    bnb_4bit_use_double_quant=True  # Double quantization (ahorra ~0.5GB VRAM)
)

print("Cargando modelo con quantización...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,  # Clasificación binaria (0=mismo autor, 1=cambio autor)
    quantization_config=bnb_config,
    device_map="auto",  # Distribución automática en GPU
    trust_remote_code=True
)

# Configurar pad token en el modelo (debe coincidir con tokenizer)
model.config.pad_token_id = tokenizer.pad_token_id

print("Modelo cuantizado cargado correctamente")
print(f"  - Compute dtype: bfloat16 (optimizado para Ada Lovelace)")

## 8. Configurar LoRA

In [ ]:
# Configurar LoRA (Low-Rank Adaptation)
# LoRA entrena matrices de bajo rango que se suman a los pesos originales
# Solo se entrenan ~0.17% de parámetros (6.8M de 4B), reduciendo VRAM y tiempo

print("Configurando LoRA...")
lora_config = LoraConfig(
    r=CONFIG["lora_r"],  # 16 (rank de matrices LoRA, dimensión de adaptación)
    lora_alpha=CONFIG["lora_alpha"],  # 32 (factor de escala, típicamente 2*r)
    target_modules=CONFIG["target_modules"],  # ["q_proj", "v_proj"] (proyecciones Q,V de atención)
    lora_dropout=CONFIG["lora_dropout"],  # 0.1 (dropout para regularización)
    bias="none",  # No entrenar bias
    task_type=TaskType.SEQ_CLS  # Sequence Classification
)

# Preparar modelo para k-bit training (congela pesos base, añade hooks para gradientes)
print("Preparando modelo para QLoRA...")
model = prepare_model_for_kbit_training(model)

# Añadir adaptadores LoRA al modelo
model = get_peft_model(model, lora_config)

# Mostrar parámetros entrenables vs totales
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nParámetros entrenables: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")

## 9. Custom Trainer con Weighted Loss

In [ ]:
import torch.nn as nn

class WeightedTrainer(Trainer):
    """
    Custom Trainer que implementa Weighted Cross-Entropy Loss.
    
    Motivación:
    - Dataset desbalanceado: 80% clase 0, 20% clase 1
    - Sin weighted loss: el modelo aprende a predecir mayormente clase 0
    - Con weighted loss: errores en clase minoritaria (1) pesan más
    
    Implementación:
    - CrossEntropyLoss con weight=[1.0, pos_weight]
    - weight[0]=1.0: clase 0 (mayoritaria) peso normal
    - weight[1]=pos_weight: clase 1 (minoritaria) peso aumentado
    - pos_weight ~4.17: errores en clase 1 cuestan 4.17x más en la loss
    
    Resultado:
    - Modelo balanceado que no ignora la clase minoritaria
    - Mejor recall en clase 1 (cambio de autor)
    """
    def __init__(self, pos_weight: float, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight
        print(f"\n[WeightedTrainer] Inicializado con pos_weight = {pos_weight:.4f}")
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """
        Sobrescribe compute_loss para usar weighted cross-entropy.
        
        Standard loss: CrossEntropyLoss()
        Weighted loss: CrossEntropyLoss(weight=[1.0, pos_weight])
        
        Efecto: Un error en clase 1 contribuye pos_weight veces más a la loss total.
        """
        labels = inputs.pop("labels")
        
        # Forward pass
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # Configurar weighted loss
        # weight[0] = 1.0 (clase 0: mismo autor, peso normal)
        # weight[1] = pos_weight (clase 1: cambio autor, peso aumentado)
        weight = torch.tensor([1.0, self.pos_weight], dtype=logits.dtype, device=logits.device)
        
        # Calcular CrossEntropyLoss con pesos
        loss_fct = nn.CrossEntropyLoss(weight=weight)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        
        return (loss, outputs) if return_outputs else loss

print("WeightedTrainer definido correctamente")

## 10. Función de Métricas

In [ ]:
# Función de métricas para evaluación
# Calcula accuracy, F1, precision y recall (macro y binario)

def compute_metrics(eval_pred):
    """
    Calcula métricas de evaluación.
    
    Métricas principales:
    - accuracy: correctitud general
    - f1_macro: promedio de F1 entre clases (mejor para datasets desbalanceados)
    - f1_class1: F1 específico de clase 1 (cambio de autor)
    - precision_macro: promedio de precision
    - recall_macro: promedio de recall
    """
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average='macro', zero_division=0)
    f1_class1 = f1_score(labels, predictions, average='binary', zero_division=0)
    precision_macro = precision_score(labels, predictions, average='macro', zero_division=0)
    recall_macro = recall_score(labels, predictions, average='macro', zero_division=0)
    
    return {
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_class1': f1_class1,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro
    }

print("Función de métricas definida")

## 11. Training Arguments (Optimizado para RTX 5070 Ti)

In [ ]:
# Training Arguments optimizados para RTX 5070 Ti (16GB VRAM)
# Balance entre velocidad (4-6 horas) y calidad (batch efectivo = 32)

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    
    # CONFIGURACIÓN DE BATCHING
    # - batch_size=16: máximo sin OOM
    # - gradient_accumulation=2: simula batch=32 acumulando gradientes
    # - eval_batch=32: evaluación más rápida (no necesita backprop)
    num_train_epochs=CONFIG['epochs'],  # 1 época
    per_device_train_batch_size=CONFIG['batch_size'],  # 16 (físico)
    per_device_eval_batch_size=CONFIG['batch_size'] * 2,  # 32 (evaluación)
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],  # 2 (batch efectivo=32)
    
    # LEARNING RATE Y OPTIMIZACIÓN
    learning_rate=CONFIG['learning_rate'],  # 2e-4 (estándar para LoRA)
    weight_decay=0.01,  # L2 regularization
    warmup_ratio=0.05,  # 5% warmup (estabiliza inicio del entrenamiento)
    
    # EVALUACIÓN Y CHECKPOINTING
    eval_strategy="steps",  # Evaluar cada N steps
    eval_steps=1000,  # Evaluar cada 1000 steps (~cada 20% del entrenamiento)
    save_strategy="steps",  # Guardar checkpoints cada N steps
    save_steps=1000,  # Guardar cada 1000 steps
    load_best_model_at_end=True,  # Cargar mejor checkpoint al final
    metric_for_best_model="f1_macro",  # Métrica de selección (mejor para desbalanceo)
    save_total_limit=2,  # Solo mantener 2 checkpoints (ahorra espacio)
    
    # PRECISION MIXTA (BF16 para Ada Lovelace)
    # BF16 tiene mismo rango que FP32 pero menos precision
    # Mejor que FP16 para evitar overflow/underflow
    bf16=True,  # Entrenar en BF16
    bf16_full_eval=True,  # Evaluar también en BF16
    
    # LOGGING
    logging_steps=50,  # Log cada 50 steps
    logging_dir=str(CHECKPOINT_DIR / "logs"),
    
    # OPTIMIZACIONES DE RENDIMIENTO
    gradient_checkpointing=False,  # OFF: más rápido pero usa más VRAM
    dataloader_pin_memory=True,  # Pin memory para transferencia GPU más rápida
    dataloader_num_workers=4,  # 4 workers paralelos para carga de datos
    dataloader_prefetch_factor=2,  # Prefetch 2 batches adelantados
    remove_unused_columns=True,  # Eliminar columnas no usadas
    
    # OPTIMIZADOR FUSIONADO
    # adamw_torch_fused: kernel CUDA fusionado (más rápido que adamw_torch)
    optim="adamw_torch_fused",
    
    # OPTIMIZACIONES ADICIONALES
    max_grad_norm=1.0,  # Gradient clipping (previene explosión de gradientes)
    ddp_find_unused_parameters=False,  # Optimización para DDP
)

# Calcular total de steps y tiempo estimado
total_samples = len(train_tokenized) if 'train_tokenized' in dir() else 156695
steps_per_epoch = total_samples // (CONFIG['batch_size'] * CONFIG['gradient_accumulation_steps'])
total_steps = steps_per_epoch * CONFIG['epochs']

print("\n" + "="*70)
print("CONFIGURACIÓN ÓPTIMA - BALANCE VELOCIDAD/CALIDAD")
print("="*70)
print(f"Batch size efectivo: {CONFIG['batch_size'] * CONFIG['gradient_accumulation_steps']}")
print(f"Total steps: {total_steps:,}")
print(f"Steps por evaluación: {training_args.eval_steps}")
print(f"Total evaluaciones: ~{total_steps // training_args.eval_steps}")
print(f"Gradient accumulation: {CONFIG['gradient_accumulation_steps']}")
print(f"BF16 full: {training_args.bf16_full_eval}")
print(f"Gradient checkpointing: {training_args.gradient_checkpointing}")
print(f"Dataloader workers: {training_args.dataloader_num_workers}")
print(f"Prefetch factor: {training_args.dataloader_prefetch_factor}")
print(f"Optimizer: {training_args.optim}")
print(f"\nTIEMPO ESTIMADO: 6-8 horas")
print(f"VRAM esperada: 12-13GB")
print(f"\nCALIDAD: GARANTIZADA (mismo batch efectivo = 32 que notebook 12)")
print("="*70)

## 12. Inicializar WeightedTrainer y Entrenar

In [ ]:
# CARGA DEL MEJOR CHECKPOINT POST-ENTRENAMIENTO
# Ejecutar esta celda si el kernel se reinició después del entrenamiento
# Lee trainer_state.json para encontrar el mejor checkpoint según F1 macro

import json

# Leer trainer_state del último checkpoint para obtener la ruta del mejor modelo
trainer_state_path = CHECKPOINT_DIR / "checkpoint-4897" / "trainer_state.json"

if trainer_state_path.exists():
    with open(trainer_state_path, 'r') as f:
        trainer_state = json.load(f)
    
    # Extraer información del mejor checkpoint
    best_checkpoint = trainer_state.get("best_model_checkpoint")
    best_metric = trainer_state.get("best_metric")
    best_step = trainer_state.get("best_global_step")
    
    print("="*70)
    print("CARGANDO MEJOR CHECKPOINT DEL ENTRENAMIENTO")
    print("="*70)
    print(f"Mejor checkpoint: {best_checkpoint}")
    print(f"Mejor F1 macro: {best_metric:.4f}")
    print(f"Step: {best_step}")
    print("="*70)
    
    # Paso 1: Recargar modelo base con quantización 4-bit
    print("\nRecargando modelo base con quantización...")
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    
    # Paso 2: Cargar adaptadores LoRA del mejor checkpoint
    # Solo se cargan los pesos de los adaptadores (~6.8M parámetros)
    # El modelo base permanece congelado y quantizado
    print(f"Cargando adaptadores LoRA desde {best_checkpoint}...")
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, best_checkpoint)
    
    # Paso 3: Recrear WeightedTrainer con el modelo cargado
    print("Recreando WeightedTrainer...")
    trainer = WeightedTrainer(
        pos_weight=pos_weight,
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
        compute_metrics=compute_metrics,
    )
    
    print("\nModelo cargado correctamente")
    print(f"Listo para evaluar")
else:
    print("Usando el modelo del entrenamiento actual (no se encontró trainer_state.json)")

## 12b. Comparar Mejor Checkpoint vs Último Checkpoint (Opcional)

In [ ]:
# COMPARACIÓN: MEJOR CHECKPOINT vs ÚLTIMO CHECKPOINT
# Objetivo: Detectar overfitting al comparar checkpoint-4000 (mejor) vs checkpoint-4897 (último)
# Si el último checkpoint es peor, el modelo sufrió overfitting después del step 4000

print("="*70)
print("EVALUANDO ÚLTIMO CHECKPOINT (step 4897)")
print("="*70)

# Paso 1: Cargar modelo base con quantización
print("\nCargando modelo base...")
model_last = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model_last.config.pad_token_id = tokenizer.pad_token_id

# Paso 2: Cargar adaptadores LoRA del último checkpoint
last_checkpoint = CHECKPOINT_DIR / "checkpoint-4897"
print(f"Cargando adaptadores LoRA desde {last_checkpoint}...")
model_last = PeftModel.from_pretrained(model_last, str(last_checkpoint))

# Paso 3: Crear trainer para evaluación
trainer_last = WeightedTrainer(
    pos_weight=pos_weight,
    model=model_last,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
)

# Paso 4: Evaluar en validation set
print("\nEvaluando último checkpoint...")
eval_result_last = trainer_last.evaluate()

# Paso 5: Comparar métricas
print("\n" + "="*70)
print("COMPARACIÓN: MEJOR vs ÚLTIMO CHECKPOINT")
print("="*70)
print(f"\n{'Métrica':<25} {'Checkpoint-4000':<20} {'Checkpoint-4897':<20} {'Diferencia':<15}")
print("-"*80)

metrics_to_compare = ['eval_accuracy', 'eval_f1_macro', 'eval_f1_class1', 
                      'eval_precision_macro', 'eval_recall_macro']

# Comparar solo si ya tenemos eval_result del mejor checkpoint
if 'eval_result' in dir():
    for metric in metrics_to_compare:
        best_val = eval_result[metric]
        last_val = eval_result_last[metric]
        diff = last_val - best_val
        diff_str = f"{diff:+.4f}"
        symbol = "MEJOR" if diff > 0 else "PEOR" if diff < 0 else "IGUAL"
        
        print(f"{metric:<25} {best_val:<20.4f} {last_val:<20.4f} {diff_str:<10} {symbol}")
else:
    for metric in metrics_to_compare:
        last_val = eval_result_last[metric]
        print(f"{metric:<25} {'N/A':<20} {last_val:<20.4f}")

print("="*70)
print("\nCONCLUSIÓN:")
if 'eval_result' in dir():
    if eval_result_last['eval_f1_macro'] < eval_result['eval_f1_macro']:
        print("El checkpoint-4000 es MEJOR que el checkpoint-4897")
        print(f"  Diferencia en F1 macro: {eval_result['eval_f1_macro'] - eval_result_last['eval_f1_macro']:.4f}")
        print("  Interpretación: El modelo sufrió overfitting después del step 4000")
    else:
        print("El checkpoint-4897 es MEJOR que el checkpoint-4000")
        print("  Interpretación: El modelo siguió mejorando después del step 4000")
else:
    print(f"F1 macro del último checkpoint: {eval_result_last['eval_f1_macro']:.4f}")
print("="*70)

## 13. Evaluar en Validation Set

In [ ]:
# Evaluación del mejor checkpoint en validation set
# Calcula métricas de rendimiento final
print("Evaluando en validation set...")
eval_result = trainer.evaluate()

print("\n" + "="*60)
print("RESULTADOS EN VALIDATION SET")
print("="*60)
for metric, value in eval_result.items():
    print(f"{metric:25s}: {value:.4f}")
print("="*60)

## 14. Guardar Modelo Final

In [ ]:
# Guardar modelo final y configuración
# Solo se guardan los adaptadores LoRA (livianos, ~27MB)
# Para inferencia: cargar modelo base + adaptadores

# Crear directorio para mejor modelo
best_model_path = CHECKPOINT_DIR / "best_model"
best_model_path.mkdir(parents=True, exist_ok=True)

# Guardar adaptadores LoRA (solo pesos entrenables)
print(f"\nGuardando adaptadores LoRA en {best_model_path}...")
model.save_pretrained(str(best_model_path))
tokenizer.save_pretrained(str(best_model_path))

# Guardar configuración completa y métricas finales
config_info = {
    "model_name": MODEL_NAME,
    "training_config": CONFIG,
    "pos_weight": float(pos_weight),  # Ratio de desbalanceo usado
    "num_train_samples": len(train_tokenized),
    "num_val_samples": len(val_tokenized),
    "class_distribution": {
        "class_0_same_author": int(num_negativos),
        "class_1_change_author": int(num_positivos)
    },
    "final_metrics": {k: float(v) for k, v in eval_result.items()}
}

with open(best_model_path / "training_info.json", "w") as f:
    json.dump(config_info, f, indent=2)

print("\nModelo y configuración guardados correctamente")
print(f"\nPath del modelo: {best_model_path}")

## 15. Evaluación Detallada por Clase

In [ ]:
# Evaluación detallada por clase
# Analiza rendimiento específico en clase 0 (mismo autor) y clase 1 (cambio autor)

# Generar predicciones en validation set
print("Generando predicciones para análisis detallado...")
predictions = trainer.predict(val_tokenized)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

# Classification report con métricas por clase
print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(
    y_true, 
    y_pred, 
    target_names=['Mismo autor (0)', 'Cambio autor (1)'],
    digits=4
))

# Matriz de confusión
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred)

print("\nMATRIZ DE CONFUSIÓN:")
print(f"                    Pred: Mismo    Pred: Cambio")
print(f"True: Mismo autor   {cm[0,0]:8d}      {cm[0,1]:8d}")
print(f"True: Cambio autor  {cm[1,0]:8d}      {cm[1,1]:8d}")
print("="*60)

## 16. Visualizar Resultados

In [ ]:
# Visualización de resultados
# Gráfica de barras con las métricas principales
metrics_to_plot = ['accuracy', 'f1_macro', 'f1_class1', 'precision_macro', 'recall_macro']
values = [eval_result[f'eval_{m}'] for m in metrics_to_plot]

plt.figure(figsize=(12, 6))
plt.bar(metrics_to_plot, values, color=['#2ecc71', '#3498db', '#e74c3c', '#f39c12', '#9b59b6'])
plt.ylim(0, 1)
plt.ylabel('Score', fontsize=12)
plt.title(f'Llama-3.1-8B Weighted Loss (pos_weight={pos_weight:.2f})', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)

# Añadir valores en las barras
for i, v in enumerate(values):
    plt.text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / 'metrics_plot.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nGráfica guardada en: {CHECKPOINT_DIR / 'metrics_plot.png'}")

## 17. Resumen Final

In [ ]:
# RESUMEN FINAL DEL ENTRENAMIENTO
# Compila toda la información clave del proceso
print("\n" + "="*70)
print(" "*20 + "RESUMEN DE ENTRENAMIENTO")
print("="*70)
print(f"\nDATASET:")
print(f"   - Estrategia: Dataset COMPLETO (sin undersampling)")
print(f"   - Train: {len(train_tokenized):,} muestras")
print(f"   - Val: {len(val_tokenized):,} muestras")
print(f"   - Desbalanceo: {pos_weight:.4f}x (clase 1 penalizada)")

print(f"\nMODELO:")
print(f"   - Base: {MODEL_NAME}")
print(f"   - Técnica: QLoRA (4-bit NF4)")
print(f"   - LoRA rank: {CONFIG['lora_r']}")
print(f"   - Target modules: {', '.join(CONFIG['target_modules'])}")
print(f"   - Parámetros entrenables: {trainable_params:,}")

print(f"\nENTRENAMIENTO:")
print(f"   - Batch size efectivo: {CONFIG['batch_size'] * CONFIG['gradient_accumulation_steps']}")
print(f"   - Learning rate: {CONFIG['learning_rate']}")
print(f"   - Épocas: {CONFIG['epochs']}")
print(f"   - Loss: Weighted Cross-Entropy (pos_weight={pos_weight:.4f})")
print(f"   - Precision: BF16")

print(f"\nRESULTADOS:")
print(f"   - Accuracy: {eval_result['eval_accuracy']:.4f}")
print(f"   - F1 Macro: {eval_result['eval_f1_macro']:.4f}")
print(f"   - F1 Class 1 (cambio): {eval_result['eval_f1_class1']:.4f}")
print(f"   - Precision Macro: {eval_result['eval_precision_macro']:.4f}")
print(f"   - Recall Macro: {eval_result['eval_recall_macro']:.4f}")

print(f"\nGUARDADO:")
print(f"   - Checkpoints: {CHECKPOINT_DIR}")
print(f"   - Mejor modelo: {best_model_path}")

print("\n" + "="*70)
print("ENTRENAMIENTO COMPLETADO EXITOSAMENTE")
print("="*70)